In [ ]:
import torch
import numpy as np
import torchvision
import matplotlib.pyplot as plt
import memtorch
from memtorch.bh.memristor.VTEAM import VTEAM
from memtorch.bh.crossbar import Crossbar
from memtorch.bh.crossbar import Scheme
from memtorch.mn.Module import patch_model
# from memtorch.bh.crossbar.parameter import Parameters
from memtorch.bh.memristor.VTEAM import VTEAM
from memtorch.bh.nonideality.NonIdeality import apply_nonidealities
from memtorch.map.Parameter import naive_map
from memtorch.map.Input import naive_scale


# MemristorHealthPredictor - MemristorHealthMonitor
# SelfHealingCrossbar - FaultTolerantNeuromorphic
# demonstrate_self_healing - run_fault_tolerance_experiment

In [ ]:
class MemristorHealthPredictor:
	"""
	Advanced health prediction system that uses temporal patterns and physics-based
	modeling to predict future memristor failures before they occur.
	"""
	def __init__(self, crossbar):
		"""
		Initialize the health predictor for a crossbar.
		
		Args:
			crossbar (memtorch.bh.crossbar.Crossbar): The crossbar to monitor
		"""
		self.crossbar = crossbar
		self.shape = crossbar.conductance_matrix.shape
		
		# Device parameters
		# self.r_on = crossbar.r_on_mean
		# self.r_off = crossbar.r_off_mean
		try:
			if hasattr(crossbar, 'memristor_model_params'):
				self.r_on = crossbar.memristor_model_params.get('r_on', 100)
				self.r_off = crossbar.memristor_model_params.get('r_off', 16000.0)
			else:
				# Default values
				self.r_on = 100
				self.r_off = 16000.0
		except:
			# Fallback to defaults
			self.r_on = 100
			self.r_off = 16000.0
		
		
		self.g_min = 1.0 / self.r_off
		self.g_max = 1.0 / self.r_on
		self.g_mid = (self.g_max + self.g_min) / 2
		self.g_range = self.g_max - self.g_min
		
		# Health tracking metrics
		self.stress = torch.zeros(self.shape)
		self.health_scores = torch.ones(self.shape) * 100
		self.write_count = torch.zeros(self.shape)
		self.read_count = torch.zeros(self.shape)
		self.stability_index = torch.ones(self.shape)  # 1.0 = stable, 0.0 = unstable
		
		# History tracking for predictive analytics
		self.g_history = [crossbar.conductance_matrix.clone()]
		self.max_history = 20  # Keep the last 20 states
		
		# Fault prediction parameters based on physics model
		self.failure_probability = torch.zeros(self.shape)
		self.failure_threshold = 0.8
		
		print(f"Initialized health predictor for {self.shape[0]}x{self.shape[1]} crossbar")
	
	def update_health_metrics(self, voltage_applied=None, write_op=False):
		"""
		Update health metrics based on recent operations and current state.
		
		Args:
			voltage_applied: Voltage matrix applied to the crossbar (optional)
			write_op: Whether this was a write operation (more stress than read)
		"""
		# Get current conductance
		current_g = self.crossbar.conductance_matrix
		
		# 1. Update operation counters
		if write_op:
			self.write_count += torch.ones(self.shape)
		else:
			self.read_count += torch.ones(self.shape)
		
			# 2. Calculate deviation from ideal values
			# Create normalized distance from middle conductance (0=middle, 1=extreme)
			g_normalized = (current_g - self.g_min) / self.g_range
			deviation_from_mid = 2 * abs(g_normalized - 0.5)
			
			# 3. Calculate conductance instability from recent history
			if len(self.g_history) >= 3:
				prev_g1 = self.g_history[-1]
				prev_g2 = self.g_history[-2]
				
				# Compute magnitude of recent changes
				recent_change1 = torch.abs(current_g - prev_g1) / self.g_range
				recent_change2 = torch.abs(prev_g1 - prev_g2) / self.g_range
				
				# Compute rate of change and acceleration
				rate_of_change = recent_change1
				change_acceleration = torch.abs(recent_change1 - recent_change2)
				
				# Update stability index - lower means less stable
				self.stability_index = self.stability_index * 0.9 + (1.0 - change_acceleration * 10) * 0.1
				self.stability_index = torch.clamp(self.stability_index, 0.0, 1.0)
			
			# 4. Update stress based on operation type
			# Write operations with large voltage swings cause more stress
			op_stress_factor = 0.02 if write_op else 0.001
			voltage_factor = 1.0
			
			if voltage_applied is not None:
				# Normalize voltage to 0-1 range
				max_voltage = self.crossbar.max_voltage if hasattr(self.crossbar, 'max_voltage') else 1.0
				voltage_normalized = torch.abs(voltage_applied) / max_voltage
				voltage_factor = 1.0 + voltage_normalized
			
			# New stress from this operation
			new_stress = op_stress_factor * voltage_factor * (0.5 + deviation_from_mid)
			
			# Stress especially increases for unstable devices
			new_stress = new_stress * (2.0 - self.stability_index)
			
			# Update accumulated stress (with partial decay over time)
			self.stress = self.stress * 0.99 + new_stress
			self.stress = torch.clamp(self.stress, 0.0, 1.0)
			
			# 5. Update health scores
			self.health_scores = 100 * (1.0 - self.stress)
			
			# 6. Store current state in history
			self.g_history.append(current_g.clone())
			if len(self.g_history) > self.max_history:
				self.g_history.pop(0)
	
	def predict_faults(self):
		"""
		Uses multiple indicators to predict which devices are likely to fail soon.
		
		Returns:
			Tensor with failure probability for each device (0-1)
		"""
		# 1. Extreme conductance values indicate potential issues
		g_normalized = (self.crossbar.conductance_matrix - self.g_min) / self.g_range
		proximity_to_extreme = torch.max(
			4 * torch.pow(g_normalized, 2),
			4 * torch.pow(1 - g_normalized, 2)
		)
		proximity_to_extreme = torch.clamp(proximity_to_extreme, 0.0, 1.0)
		
		# 2. Instability is a strong predictor of failure
		instability_factor = 1.0 - self.stability_index
		
		# 3. Recent conductance drift trend
		drift_factor = torch.zeros(self.shape)
		if len(self.g_history) >= 5:
			# Calculate average drift direction and magnitude over recent history
			g_diffs = []
			for i in range(1, min(5, len(self.g_history))):
				g_diff = (self.g_history[-i] - self.g_history[-i-1]) / self.g_range
				g_diffs.append(g_diff)
			
			# Convert list to tensor and calculate statistics
			g_diffs_tensor = torch.stack(g_diffs)
			drift_magnitude = torch.mean(torch.abs(g_diffs_tensor), dim=0)
			
			# Check if drift is consistent (same direction)
			drift_consistency = torch.std(torch.sign(g_diffs_tensor), dim=0)
			consistent_drift = drift_consistency < 0.5
			
			# High consistent drift is concerning
			drift_factor = drift_magnitude * (2.0 - drift_consistency)
			drift_factor = torch.clamp(drift_factor * 5.0, 0.0, 1.0)  # Scale and clamp
		
		# 4. Stress history (accumulated wear)
		stress_factor = self.stress
		
		# Combine factors with weights to get failure probability
		self.failure_probability = (
			0.4 * stress_factor +
			0.3 * instability_factor +
			0.2 * proximity_to_extreme +
			0.1 * drift_factor
		)
		
		# Return the updated failure probability
		return self.failure_probability
	
	def get_at_risk_devices(self, threshold=None):
		"""
		Get a mask of devices at risk of failure.
		
		Args:
			threshold: Failure probability threshold (default: self.failure_threshold)
			
		Returns:
			Boolean tensor mask of at-risk devices
		"""
		if threshold is None:
			threshold = self.failure_threshold
			
		return self.failure_probability > threshold
	
	def visualize_health(self):
		"""
		Visualize health metrics and failure predictions.
		"""
		# Create figure with multiple subplots
		fig, axes = plt.subplots(2, 2, figsize=(15, 12))
		
		# 1. Health scores
		im1 = axes[0, 0].imshow(self.health_scores.numpy(), cmap='RdYlGn', vmin=0, vmax=100)
		axes[0, 0].set_title('Health Scores')
		fig.colorbar(im1, ax=axes[0, 0])
		
		# 2. Stability index
		im2 = axes[0, 1].imshow(self.stability_index.numpy(), cmap='coolwarm', vmin=0, vmax=1)
		axes[0, 1].set_title('Stability Index')
		fig.colorbar(im2, ax=axes[0, 1])
		
		# 3. Failure probability
		im3 = axes[1, 0].imshow(self.failure_probability.numpy(), cmap='plasma', vmin=0, vmax=1)
		axes[1, 0].set_title('Failure Probability')
		fig.colorbar(im3, ax=axes[1, 0])
		
		# 4. Current conductance
		g_normalized = ((self.crossbar.conductance_matrix - self.g_min) / self.g_range).numpy()
		im4 = axes[1, 1].imshow(g_normalized, cmap='viridis', vmin=0, vmax=1)
		axes[1, 1].set_title('Normalized Conductance')
		fig.colorbar(im4, ax=axes[1, 1])
		
		plt.tight_layout()
		plt.show()
	
	def get_health_metrics(self):
		"""
		Get summary health metrics for the crossbar.
		
		Returns:
			Dictionary of health metrics
		"""
		# Calculate probabilities of failure within different timeframes
		imminent_failures = (self.failure_probability > 0.8)
		upcoming_failures = (self.failure_probability > 0.5) & (self.failure_probability <= 0.8)
		at_risk_devices = (self.failure_probability > 0.3) & (self.failure_probability <= 0.5)
		
		# Count devices in each category
		imminent_count = imminent_failures.sum().item()
		upcoming_count = upcoming_failures.sum().item()
		at_risk_count = at_risk_devices.sum().item()
		total_devices = self.shape[0] * self.shape[1]
		
		# Calculate average values
		avg_health = self.health_scores.mean().item()
		avg_stability = self.stability_index.mean().item()
		avg_failure_prob = self.failure_probability.mean().item()
		
		# Get conductance statistics
		g_values = self.crossbar.conductance_matrix
		g_normalized = (g_values - self.g_min) / self.g_range
		g_hist, _ = np.histogram(g_normalized.numpy().flatten(), bins=10, range=(0, 1))
		
		# Check if any devices are actually failed (at extreme conductance values)
		g_min_threshold = self.g_min + 0.01 * self.g_range
		g_max_threshold = self.g_max - 0.01 * self.g_range
		
		stuck_at_min = (g_values <= g_min_threshold).sum().item()
		stuck_at_max = (g_values >= g_max_threshold).sum().item()
		total_stuck = stuck_at_min + stuck_at_max
		
		return {
			'avg_health': avg_health,
			'avg_stability': avg_stability,
			'avg_failure_prob': avg_failure_prob,
			'imminent_failures': imminent_count,
			'imminent_percent': 100 * imminent_count / total_devices,
			'upcoming_failures': upcoming_count,
			'upcoming_percent': 100 * upcoming_count / total_devices,
			'at_risk_count': at_risk_count,
			'at_risk_percent': 100 * at_risk_count / total_devices,
			'total_at_risk': imminent_count + upcoming_count + at_risk_count,
			'total_at_risk_percent': 100 * (imminent_count + upcoming_count + at_risk_count) / total_devices,
			'conductance_distribution': g_hist,
			'stuck_at_min': stuck_at_min,
			'stuck_at_max': stuck_at_max,
			'total_stuck': total_stuck,
			'stuck_percent': 100 * total_stuck / total_devices
		}

In [ ]:
class MemristorHealthMonitor:
    """
    Health monitoring system for memristive crossbars, working with observable properties.
    """
    
    def __init__(self, crossbar):
        """
        Initialize the health monitor for a crossbar.
        
        Args:
            crossbar: The crossbar to monitor
        """
        self.crossbar = crossbar
        self.shape = crossbar.conductance_matrix.shape
        
        # Use visible properties instead of assuming internal ones
        try:
            # Access r_on and r_off from crossbar if available
            if hasattr(crossbar, 'r_on'):
                self.r_on = crossbar.r_on
                self.r_off = crossbar.r_off
            elif hasattr(crossbar, 'memristor_model_params'):
                self.r_on = crossbar.memristor_model_params.get('r_on', 100)
                self.r_off = crossbar.memristor_model_params.get('r_off', 10000)
            else:
                # Default values
                self.r_on = 100
                self.r_off = 10000
        except:
            # Fallback to defaults
            self.r_on = 100
            self.r_off = 10000
        
        # Derived properties
        self.g_min = 1.0 / self.r_off
        self.g_max = 1.0 / self.r_on
        self.g_range = self.g_max - self.g_min
        
        # Health tracking metrics - these don't rely on internal properties
        self.health_scores = torch.ones(self.shape) * 100  # 100 = perfect health
        self.operation_counts = torch.zeros(self.shape)    # Count operations per device
        
        # History tracking for simple analysis
        self.g_history = []  # Store last few states
        self.max_history = 5
        self.update_history()
        
        print(f"Initialized health monitor for {self.shape[0]}x{self.shape[1]} crossbar")
    
    def update_history(self):
        """Update conductance history."""
        current_g = self.crossbar.conductance_matrix.clone()
        self.g_history.append(current_g)
        if len(self.g_history) > self.max_history:
            self.g_history.pop(0)
    
    def update_health_metrics(self, write_op=False):
        """
        Update health metrics based on current conductance state.
        
        Args:
            write_op: Whether this was a write operation (more stressful than read)
        """
        # Get current conductance
        current_g = self.crossbar.conductance_matrix
        
        # Add to history
        self.update_history()
        
        # Update operation count
        self.operation_counts += 1
        
        # Calculate health based on how close devices are to extreme values
        g_normalized = (current_g - self.g_min) / self.g_range
        g_normalized = torch.clamp(g_normalized, 0, 1)  # Ensure in range [0,1]
        
        # Calculate deviation from middle conductance (0.5 = ideal middle)
        # Larger deviation = potentially less healthy
        deviation = torch.abs(g_normalized - 0.5) * 2  # Scale to [0,1]
        
        # Update health score based on deviation
        # Small penalty for read operations, larger for write
        penalty = 0.02 if write_op else 0.001
        self.health_scores = torch.clamp(self.health_scores - deviation * penalty, 0, 100)
        
        # If we have history, check for instability
        if len(self.g_history) >= 2:
            # Calculate conductance change rate
            prev_g = self.g_history[-2]
            change_rate = torch.abs(current_g - prev_g) / self.g_range
            
            # Large changes might indicate instability
            unstable = change_rate > 0.1
            self.health_scores[unstable] -= 0.5  # Additional penalty
        
        # Ensure health scores stay in valid range
        self.health_scores = torch.clamp(self.health_scores, 0, 100)
    
    def detect_faults(self, threshold=0.05):
        """
        Detect potential faults based on conductance values and health scores.
        
        Args:
            threshold: Threshold for considering a device faulty
            
        Returns:
            Boolean tensor mask of detected faults
        """
        # Get current conductance
        current_g = self.crossbar.conductance_matrix
        
        # Calculate how close each device is to min/max conductance
        g_normalized = (current_g - self.g_min) / self.g_range
        g_normalized = torch.clamp(g_normalized, 0, 1)
        
        # Devices very close to min/max might be stuck
        close_to_min = g_normalized < threshold
        close_to_max = g_normalized > (1 - threshold)
        potentially_stuck = close_to_min | close_to_max
        
        # Low health score also indicates potential issues
        low_health = self.health_scores < 20
        
        # Combine indicators
        return potentially_stuck | low_health
    
    def get_critical_devices(self, health_threshold=10, extreme_threshold=0.02):
        """
        Get devices in critical condition that should be prioritized for healing.
        
        Args:
            health_threshold: Health score below this is critical
            extreme_threshold: How close to min/max conductance is critical
            
        Returns:
            List of (row, col) tuples of critical devices
        """
        # Get current conductance
        current_g = self.crossbar.conductance_matrix
        
        # Calculate normalized conductance
        g_normalized = (current_g - self.g_min) / self.g_range
        g_normalized = torch.clamp(g_normalized, 0, 1)
        
        # Find critical devices
        critical_devices = []
        
        for i in range(self.shape[0]):
            for j in range(self.shape[1]):
                # Check if health is critical
                if self.health_scores[i, j] < health_threshold:
                    critical_devices.append((i, j))
                    continue
                
                # Check if conductance is extreme
                g_norm = g_normalized[i, j].item()
                if g_norm < extreme_threshold or g_norm > (1 - extreme_threshold):
                    critical_devices.append((i, j))
        
        return critical_devices
    
    def get_health_summary(self):
        """
        Get a summary of crossbar health.
        
        Returns:
            Dictionary with health metrics
        """
        # Calculate statistics
        avg_health = self.health_scores.mean().item()
        min_health = self.health_scores.min().item()
        
        # Get current conductance
        current_g = self.crossbar.conductance_matrix
        g_normalized = (current_g - self.g_min) / self.g_range
        g_normalized = torch.clamp(g_normalized, 0, 1)
        
        # Count potentially stuck devices
        stuck_at_min = (g_normalized < 0.05).sum().item()
        stuck_at_max = (g_normalized > 0.95).sum().item()
        
        return {
            'avg_health': avg_health,
            'min_health': min_health,
            'stuck_at_min': stuck_at_min,
            'stuck_at_max': stuck_at_max,
            'total_stuck': stuck_at_min + stuck_at_max,
            'critical_count': (self.health_scores < 20).sum().item(),
            'total_operations': self.operation_counts.mean().item()
        }
    
    def visualize_health(self):
        """
        Visualize health metrics.
        """
        plt.figure(figsize=(12, 4))
        
        # Health scores
        plt.subplot(1, 3, 1)
        plt.imshow(self.health_scores.numpy(), cmap='RdYlGn', vmin=0, vmax=100)
        plt.colorbar(label='Health Score')
        plt.title('Device Health')
        
        # Conductance values
        plt.subplot(1, 3, 2)
        current_g = self.crossbar.conductance_matrix
        g_normalized = (current_g - self.g_min) / self.g_range
        g_normalized = torch.clamp(g_normalized, 0, 1)
        plt.imshow(g_normalized.numpy(), cmap='viridis')
        plt.colorbar(label='Normalized Conductance')
        plt.title('Conductance Values')
        
        # Operation count
        plt.subplot(1, 3, 3)
        plt.imshow(self.operation_counts.numpy(), cmap='Blues')
        plt.colorbar(label='Operations')
        plt.title('Device Usage')
        
        plt.tight_layout()
        plt.show()

In [ ]:
class SelfHealingCrossbar:
    """
    Self-healing memristive crossbar that works with MemTorch's implementation.
    Uses a simpler, more compatible approach for redundancy management.
    """
    
    def __init__(self, memristor_model=VTEAM, shape=(128, 128), redundancy_factor=0.1, tile_shape=None):
        """
        Initialize a self-healing crossbar with built-in redundancy.
        
        Args:
            memristor_model: MemTorch memristor model to use
            shape: Crossbar dimensions (rows, cols)
            redundancy_factor: Percentage of redundant devices (0.1 = 10%)
        """
        # Store original dimensions
        self.original_shape = shape
        self.redundancy_factor = redundancy_factor
        
        # Calculate effective shape with redundancy
        effective_rows = int(shape[0] * (1 + redundancy_factor))
        effective_cols = int(shape[1] * (1 + redundancy_factor))
        self.effective_shape = (effective_rows, effective_cols)
        
        # Initialize tracking of which devices are redundant vs primary
        self.is_redundant = torch.zeros(self.effective_shape, dtype=torch.bool)
        self.is_redundant[shape[0]:, :] = True  # Rows beyond original are redundant
        self.is_redundant[:, shape[1]:] = True  # Columns beyond original are redundant
        
        # Track remapped devices (logical device -> physical device mapping)
        self.remapping = {}  # (logical_row, logical_col) -> (physical_row, physical_col)
        
        # Keep track of available redundant devices
        self.available_redundant = []
        for i in range(self.effective_shape[0]):
            for j in range(self.effective_shape[1]):
                if self.is_redundant[i, j]:
                    self.available_redundant.append((i, j))
        
        # Initialize memristor parameters with safe defaults from the documentation
        memristor_params = {
            'r_on': 100,      # Low resistance state (Ohms)
            'r_off': 10000,   # High resistance state (Ohms)
            'time_series_resolution': 1e-10  # Required parameter
        }
        
        # Add model-specific parameters if VTEAM
        if memristor_model.__name__ == 'VTEAM':
            memristor_params.update({
                'd': 3e-9,        # Device length (m)
                'k_on': -10,      # k_on parameter
                'k_off': 5e-4,    # k_off parameter
                'alpha_on': 3,    # alpha_on parameter
                'alpha_off': 1,   # alpha_off parameter
                'v_on': -0.2,     # Positive write threshold voltage (V)
                'v_off': 0.02,    # Negative write threshold voltage (V)
                'x_on': 0,        # x_on parameter
                'x_off': 3e-9,    # x_off parameter
            })
        
        # Initialize the crossbar using MemTorch
        self.crossbar = Crossbar(
            memristor_model=memristor_model,
            memristor_model_params=memristor_params, 
            shape=self.effective_shape,
            tile_shape=tile_shape
        )
        
        # Track health metrics without assuming internal properties
        self.r_on = memristor_params['r_on']
        self.r_off = memristor_params['r_off']
        self.g_min = 1.0 / self.r_off
        self.g_max = 1.0 / self.r_on
        
        # Statistics tracking
        self.mitigation_count = 0
        self.remapping_count = 0
        self.operation_count = 0
        
        print(f"Created self-healing crossbar with shape {shape} and {len(self.available_redundant)} redundant devices")
    
    # Inside SelfHealingCrossbar class
    def _map_logical_to_physical(self, logical_row, logical_col):
        """
        Maps logical (original) coordinates to physical (effective) coordinates.
        
        Args:
            logical_row: Row in the original matrix
            logical_col: Column in the original matrix
            
        Returns:
            Tuple (physical_row, physical_col) for accessing the crossbar matrix
        """
        if logical_row >= self.original_shape[0] or logical_col >= self.original_shape[1]:
            raise ValueError(f"Logical coordinates ({logical_row},{logical_col}) out of bounds for original shape {self.original_shape}")
        
        if hasattr(self, 'is_remapped') and self.is_remapped[logical_row, logical_col]:
            # This device has been remapped, find its physical location
            idx = self.device_mapping[logical_row, logical_col]
            physical_row = idx // self.effective_shape[1]
            physical_col = idx % self.effective_shape[1]
        else:
            # Not remapped, use the same coordinates
            physical_row = logical_row
            physical_col = logical_col
            
        # Add bounds check to prevent indexing errors
        if physical_row >= self.effective_shape[0] or physical_col >= self.effective_shape[1]:
            print(f"Warning: Mapped coordinates ({physical_row},{physical_col}) out of bounds, using original")
            return logical_row, logical_col
            
        return physical_row, physical_col
    
    def visualize_health_and_mapping(self):
        """
        Visualize the health and mapping of devices in the crossbar.
        """
        plt.figure(figsize=(12, 4))
        
        # 1. Show conductance matrix
        plt.subplot(1, 3, 1)
        g_normalized = (self.crossbar.conductance_matrix - self.g_min) / (self.g_max - self.g_min)
        g_normalized = torch.clamp(g_normalized, 0, 1)
        plt.imshow(g_normalized.numpy(), cmap='viridis')
        plt.colorbar(label='Normalized Conductance')
        plt.title('Conductance Values')
        
        # 2. Show remapped devices
        plt.subplot(1, 3, 2)
        remapping_mask = torch.zeros(self.effective_shape)
        
        # Mark original area
        for i in range(self.original_shape[0]):
            for j in range(self.original_shape[1]):
                if (i, j) in self.remapping:
                    physical_row, physical_col = self.remapping[(i, j)]
                    remapping_mask[physical_row, physical_col] = 2  # Remapped destination
                    remapping_mask[i, j] = 1  # Original position (remapped source)
                else:
                    remapping_mask[i, j] = 0.5  # Normal original device
        
        # Mark redundant area
        for i in range(self.effective_shape[0]):
            for j in range(self.effective_shape[1]):
                if self.is_redundant[i, j] and remapping_mask[i, j] == 0:
                    remapping_mask[i, j] = 0.25  # Unused redundant
        
        plt.imshow(remapping_mask.numpy(), cmap='coolwarm')
        plt.colorbar(label='Device Status')
        plt.title('Device Remapping')
        
        # 3. Show health estimate
        plt.subplot(1, 3, 3)
        # Estimate health based on how close conductances are to extremes
        g_normalized = (self.crossbar.conductance_matrix - self.g_min) / (self.g_max - self.g_min)
        g_normalized = torch.clamp(g_normalized, 0, 1)
        deviation = torch.abs(g_normalized - 0.5) * 2
        health_scores = 100 * (1.0 - deviation)
        
        plt.imshow(health_scores.numpy(), cmap='RdYlGn', vmin=0, vmax=100)
        plt.colorbar(label='Health Score')
        plt.title('Estimated Device Health')
        
        plt.tight_layout()
        plt.show()
    
    def write_conductance_matrix(self, conductance_matrix):
        """
        Write a conductance matrix to the crossbar, accounting for any remapped devices.
        
        Args:
            conductance_matrix: Conductance matrix to write (shape should match original_shape)
        """
        if conductance_matrix.shape != self.original_shape:
            raise ValueError(f"Expected conductance matrix of shape {self.original_shape}, got {conductance_matrix.shape}")
        
        # Create a full conductance matrix for the crossbar, initialized with zeros
        effective_conductance = torch.zeros(self.effective_shape)
        
        # Copy the original values to their appropriate location (remapped or not)
        for i in range(self.original_shape[0]):
            for j in range(self.original_shape[1]):
                
                physical_row, physical_col = self._map_logical_to_physical(i, j)
                # Set the conductance value at the physical position
                effective_conductance[physical_row, physical_col] = conductance_matrix[i, j]
        
        # Write to the crossbar
        self.crossbar.write_conductance_matrix(effective_conductance)
        self.operation_count += 1
    
    def read_conductance_matrix(self):
        """
        Read the conductance matrix from the crossbar, accounting for remapped devices.
        
        Returns:
            Conductance matrix with original dimensions
        """
        # Get the full conductance matrix from the crossbar
        full_conductance = self.crossbar.conductance_matrix
        
        # Create a matrix to hold the logical values
        logical_conductance = torch.zeros(self.original_shape)
        
        # Copy values from their physical positions to logical positions
        for i in range(self.original_shape[0]):
            for j in range(self.original_shape[1]):
                logical_pos = (i, j)
                
                physical_row, physical_col = self._map_logical_to_physical(i, j)
                
                # Get the conductance value from the physical position
                logical_conductance[i, j] = full_conductance[physical_row, physical_col]
        
        return logical_conductance
    
    def simulate_matmul(self, input_tensor):
        """
        Perform matrix multiplication, taking remapping into account.
        
        Args:
            input_tensor: Input tensor for the multiplication
            
        Returns:
            Output tensor from the matrix multiplication
        """
        # Check dimensions
        if input_tensor.shape[1] != self.original_shape[0]:
            raise ValueError(f"Input shape mismatch: expected {self.original_shape[0]}, got {input_tensor.shape[1]}")
        
        # Create input tensor with extended dimensions to account for redundant rows
        extended_input = torch.zeros((input_tensor.shape[0], self.effective_shape[0]))
        
        # Copy the original inputs to their correct positions
        for i in range(self.original_shape[0]):
            # Find all logical positions that map to this input row
            extended_input[:, i] = input_tensor[:, i]
            
            # Check for any remappings from this row to redundant rows
            for j in range(self.original_shape[1]):
                logical_pos = (i, j)
                if logical_pos in self.remapping:
                    physical_row, _ = self._map_logical_to_physical(i, j)
                    if physical_row >= self.original_shape[0]:  # If remapped to redundant row
                        extended_input[:, physical_row] = input_tensor[:, i]
        
        # Perform matrix multiplication
        full_output = self.crossbar.simulate_matmul(input=extended_input)
        
        # Extract only the relevant columns, accounting for remapping
        output = torch.zeros((input_tensor.shape[0], self.original_shape[1]))
        
        # Copy values from physical to logical positions
        for j in range(self.original_shape[1]):
            column_mapped = False
            
            # Check if any logical position has been remapped to a redundant column
            for i in range(self.original_shape[0]):
                logical_pos = (i, j)
                if logical_pos in self.remapping:
                    _, physical_col = self._map_logical_to_physical(i, j)
                    if physical_col >= self.original_shape[1]:  # If remapped to redundant column
                        output[:, j] = full_output[:, physical_col]
                        column_mapped = True
                        break
            
            # If not remapped, use the original column
            if not column_mapped:
                output[:, j] = full_output[:, j]
        
        self.operation_count += 1
        return output
    
    def remap_device(self, logical_row, logical_col):
        """
        Remap a device to an available redundant device.
        
        Args:
            logical_row: Row of the device to remap
            logical_col: Column of the device to remap
            
        Returns:
            True if remapping successful, False otherwise
        """
        logical_pos = (logical_row, logical_col)
        
        # Check if already remapped
        if logical_pos in self.remapping:
            return False
        
        # Check if we have redundant devices available
        if not self.available_redundant:
            return False
        
        # Get current conductance
        full_conductance = self.crossbar.conductance_matrix
        current_conductance = full_conductance[logical_row, logical_col]
        
        # Get an available redundant device
        physical_row, physical_col = self.available_redundant.pop(0)
        
        # Update mapping
        self.remapping[logical_pos] = (physical_row, physical_col)
        
        # Copy conductance to the redundant device
        self.crossbar.conductance_matrix[physical_row, physical_col] = current_conductance
        
        self.remapping_count += 1
        return True
    
    def inject_faults(self, lrs_proportion=0.01, hrs_proportion=0.01):
        """
        Inject stuck-at faults into the crossbar for testing.
        
        Args:
            lrs_proportion: Proportion of devices to make stuck at LRS
            hrs_proportion: Proportion of devices to make stuck at HRS
            
        Returns:
            Number of faults injected
        """
        # Get original crossbar size
        rows, cols = self.original_shape
        total_devices = rows * cols
        
        # Calculate number of devices to make stuck
        num_lrs_faults = int(lrs_proportion * total_devices)
        num_hrs_faults = int(hrs_proportion * total_devices)
        
        # Create random indices for stuck devices
        device_indices = torch.randperm(total_devices)
        lrs_indices = device_indices[:num_lrs_faults]
        hrs_indices = device_indices[num_lrs_faults:num_lrs_faults+num_hrs_faults]
        
        # Apply faults
        faults_injected = 0
        
        for idx in lrs_indices:
            row = idx // cols
            col = idx % cols
            
            # Skip already remapped devices
            if (row, col) in self.remapping:
                continue
                
            # Make device stuck at LRS
            self.crossbar.conductance_matrix[row, col] = self.g_max
            faults_injected += 1
        
        for idx in hrs_indices:
            row = idx // cols
            col = idx % cols
            
            # Skip already remapped devices
            if (row, col) in self.remapping:
                continue
                
            # Make device stuck at HRS
            self.crossbar.conductance_matrix[row, col] = self.g_min
            faults_injected += 1
        
        print(f"Injected {faults_injected} faults ({num_lrs_faults} LRS, {num_hrs_faults} HRS)")
        return faults_injected
    
    def detect_faults(self, threshold=0.1):
        """
        Detect potential faults in the crossbar based on extreme conductance values.
        
        Args:
            threshold: How close to g_min or g_max to consider a fault
            
        Returns:
            List of detected faults as (row, col) tuples
        """
        # Get current conductance matrix
        conductance = self.crossbar.conductance_matrix
        
        # Define thresholds for detecting stuck devices
        g_min_threshold = self.g_min + threshold * (self.g_max - self.g_min)
        g_max_threshold = self.g_max - threshold * (self.g_max - self.g_min)
        
        # Find potential faults
        faults = []
        
        for i in range(self.original_shape[0]):
            for j in range(self.original_shape[1]):
                # Skip already remapped devices
                if (i, j) in self.remapping:
                    continue
                
                g_value = conductance[i, j]
                
                # Check if device appears stuck
                if g_value <= g_min_threshold or g_value >= g_max_threshold:
                    faults.append((i, j))
        
        return faults
    
    def apply_self_healing(self):
        """
        Apply self-healing by detecting and remapping faulty devices.
        
        Returns:
            Number of devices healed
        """
        # Skip if no redundant devices available
        if not self.available_redundant:
            return 0
        
        # Detect potential faults
        faults = self.detect_faults()
        if not faults:
            return 0
        
        # Apply remapping to as many faults as possible
        healed_count = 0
        
        for row, col in faults:
            if healed_count >= len(self.available_redundant):
                break
                
            if self.remap_device(row, col):
                healed_count += 1
        
        self.mitigation_count += healed_count
        print(f"Healed {healed_count} devices. Total remapped: {self.remapping_count}")
        
        return healed_count

In [ ]:
def demonstrate_self_healing():
    """
    Demonstrate the self-healing capabilities of the crossbar
    """
    # Create a self-healing crossbar
    crossbar_shape = (64, 64)  # Smaller for demonstration
    healing_crossbar = SelfHealingCrossbar(
        shape=crossbar_shape,
        redundancy_factor=0.2  # 20% redundancy
    )
    
    # Initialize with random conductance values
    g_min = 1 / 10000  # 1/R_off
    g_max = 1 / 100    # 1/R_on
    random_conductance = torch.zeros(crossbar_shape).uniform_(g_min, g_max)
    healing_crossbar.write_conductance_matrix(random_conductance)
    
    # Print initial health report
    print("\nInitial health report:")
    report = healing_crossbar.get_health_report()
    print(f"Average health: {report['avg_health']:.2f}%")
    print(f"Devices at risk: {report['total_at_risk']} ({report['total_at_risk_percent']:.2f}%)")
    
    # Inject some faults to simulate aging and wear
    # We'll create a pattern of gradually increasing faults
    print("\nSimulating device aging and wear...")
    
    # Run a series of operations with increasing stress
    num_iterations = 100
    for i in range(num_iterations):
        # Every 10 iterations, print a status update
        if i % 10 == 0:
            print(f"Iteration {i}/{num_iterations}")
            
        # Create a batch of random inputs
        batch_size = 32
        input_tensor = torch.randn(batch_size, crossbar_shape[0])
        
        # Simulate matrix multiplication (read operation)
        output = healing_crossbar.simulate_matmul(input_tensor)
        
        # Every 5 iterations, perform a write operation (more stressful)
        if i % 5 == 0:
            # Generate some updates to the conductance matrix
            update_mask = torch.rand(crossbar_shape) < 0.2  # Update 20% of values
            updates = torch.zeros(crossbar_shape).uniform_(g_min, g_max)
            
            # Read current conductance values
            current_g = torch.zeros(crossbar_shape)
            for r in range(crossbar_shape[0]):
                for c in range(crossbar_shape[1]):
                    if not healing_crossbar.is_remapped[r, c]:
                        current_g[r, c] = healing_crossbar.crossbar.conductance_matrix[r, c]
                    else:
                        idx = healing_crossbar.device_mapping[r, c]
                        row = idx // healing_crossbar.effective_shape[1]
                        col = idx % healing_crossbar.effective_shape[1]
                        current_g[r, c] = healing_crossbar.crossbar.conductance_matrix[row, col]
            
            # Apply updates selectively
            new_g = current_g.clone()
            new_g[update_mask] = updates[update_mask]
            
            # Write the updated conductance matrix
            healing_crossbar.write_conductance_matrix(new_g)
        
        # Every 20 iterations, deliberately inject some faults
        if i % 20 == 0 and i > 0:
            # Calculate how many devices to make faulty
            fault_percentage = min(2.0 + i/20, 10.0)  # Gradually increase to max 10%
            num_faults = int(crossbar_shape[0] * crossbar_shape[1] * fault_percentage / 100)
            
            print(f"Injecting {num_faults} artificial faults ({fault_percentage:.1f}% of devices)")
            
            # Create random fault locations
            fault_indices = torch.randperm(crossbar_shape[0] * crossbar_shape[1])[:num_faults]
            fault_rows = fault_indices // crossbar_shape[1]
            fault_cols = fault_indices % crossbar_shape[1]
            
            # Apply stress to these devices
            for idx in range(num_faults):
                row, col = fault_rows[idx].item(), fault_cols[idx].item()
                
                if not healing_crossbar.is_remapped[row, col]:
                    # Increase stress for this device
                    healing_crossbar.health_predictor.stress[row, col] = min(
                        healing_crossbar.health_predictor.stress[row, col] + 0.3,
                        1.0
                    )
                    # Decrease stability
                    healing_crossbar.health_predictor.stability_index[row, col] = max(
                        healing_crossbar.health_predictor.stability_index[row, col] - 0.4,
                        0.0
                    )
                    
                    # Update health score
                    healing_crossbar.health_predictor.health_scores[row, col] = (
                        100 * (1.0 - healing_crossbar.health_predictor.stress[row, col])
                    )
            
            # Force a health check and mitigation
            healing_crossbar.apply_self_healing(force_check=True)
    
    # Print final health report
    print("\nFinal health report:")
    report = healing_crossbar.get_health_report(detailed=True)
    print(f"Average health: {report['avg_health']:.2f}%")
    print(f"Devices at risk: {report['total_at_risk']} ({report['total_at_risk_percent']:.2f}%)")
    print(f"Remapped devices: {report['remapping_count']}")
    print(f"Total mitigations: {report['mitigation_count']}")
    print(f"Remaining redundant devices: {report['redundant_devices']}")
    
    if 'healing_efficiency' in report:
        print(f"Healing efficiency: {report['healing_efficiency']:.2f}")
    
    if 'estimated_remaining_percentage' in report and report['estimated_remaining_percentage'] < float('inf'):
        print(f"Estimated remaining lifetime: {report['estimated_remaining_percentage']:.2f}% of current lifetime")
    
    # Visualize the final state
    healing_crossbar.visualize_health_and_mapping()
    
    return healing_crossbar

In [ ]:
class SimpleCNN(torch.nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = torch.nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = torch.nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc1 = torch.nn.Linear(32 * 8 * 8, 128)
        self.fc2 = torch.nn.Linear(128, 10)
        self.relu = torch.nn.ReLU()
        self.pool = torch.nn.MaxPool2d(2, 2)
        self.dropout = torch.nn.Dropout(0.3)

    def forward(self, x):
        """
        Forward pass with fault monitoring.
        
        Args:
            x: Input tensor
            
        Returns:
            Model output
        """
        # Forward pass using the memristive model
        with torch.set_grad_enabled(self.memristive_model.training):
            output = self.memristive_model(x)
        
        # Update stats
        self.total_operations += 1
        
        # Periodically check health
        if self.total_operations % 10 == 0:
            with torch.no_grad():  # Don't affect gradients
                self.check_health()
        
        return output

In [ ]:
class FaultTolerantNeuromorphic:
    """
    Fault-tolerant neuromorphic computing system that works within MemTorch's constraints.
    Uses a simpler approach that preserves gradient flow.
    """
    
    def __init__(self, base_model=None, memristor_model=VTEAM, redundancy_factor=0.1):
        """
        Initialize the fault-tolerant neuromorphic system.
        
        Args:
            base_model: Base PyTorch model (if None, creates a SimpleCNN)
            memristor_model: Memristor model to use
            redundancy_factor: Redundancy factor for self-healing crossbars
        """
        # Create base model if not provided
        if base_model is None:
            self.base_model = SimpleCNN()
        else:
            self.base_model = base_model
        
        # Define memristor parameters
        memristor_params = {
            'r_on': 100,      # Low resistance state (Ohms)
            'r_off': 10000,   # High resistance state (Ohms)
            'time_series_resolution': 1e-10  # Required parameter
        }
        
        # Add model-specific parameters if using VTEAM
        if memristor_model.__name__ == 'VTEAM':
            memristor_params.update({
                'd': 3e-9,        # Device length (m)
                'k_on': -10,      # k_on parameter
                'k_off': 5e-4,    # k_off parameter
                'alpha_on': 3,    # alpha_on parameter
                'alpha_off': 1,   # alpha_off parameter
                'v_on': -0.2,     # Positive write threshold voltage (V)
                'v_off': 0.02,    # Negative write threshold voltage (V)
                'x_on': 0,        # x_on parameter
                'x_off': 3e-9,    # x_off parameter
            })
        
        # Convert to memristive model
        # Only patch linear layers for simplicity
        try:
            self.memristive_model = patch_model(
                model=self.base_model,
                memristor_model=memristor_model,
                memristor_model_params=memristor_params,
                module_parameters_to_patch=[torch.nn.Linear],  # Start with just linear layers
                mapping_routine=naive_map,
                transistor=True,
                scheme=Scheme.DoubleColumn,
                max_input_voltage=0.3,
                scaling_routine=naive_scale,
                ADC_resolution=8,
                ADC_overflow_rate=0.0,
                quant_method='linear'
            )
            print("Successfully patched model with memristors")
        except Exception as e:
            print(f"Error patching model: {e}")
            # Fall back to original model
            self.memristive_model = self.base_model
            print("Using original model as fallback")
        
        # Track memristive layers for monitoring
        self.memristive_layers = {}  # name -> module mapping
        self.health_monitors = {}    # name -> health monitor mapping
        
        # Scan for memristive layers
        for name, module in self.memristive_model.named_modules():
            if hasattr(module, 'crossbars'):
                print(f"Found memristive layer: {name}")
                self.memristive_layers[name] = module
                
                # Create health monitors for each crossbar
                for i, crossbar in enumerate(module.crossbars):
                    crossbar_name = f"{name}_crossbar_{i}"
                    self.health_monitors[crossbar_name] = MemristorHealthMonitor(crossbar)
        
        # Create backup of original weights for critical situations
        self.original_weights = {}
        for name, module in self.base_model.named_modules():
            if isinstance(module, torch.nn.Linear) or isinstance(module, torch.nn.Conv2d):
                if hasattr(module, 'weight'):
                    self.original_weights[name] = module.weight.data.clone()
        
        # Statistics
        self.total_operations = 0
        self.total_faults_detected = 0
        self.total_mitigations = 0
        
        print(f"Initialized fault-tolerant system with {len(self.memristive_layers)} memristive layers")
    
    def forward(self, x):
        """
        Forward pass with fault monitoring.
        
        Args:
            x: Input tensor
            
        Returns:
            Model output
        """
        # Forward pass using the memristive model
        output = self.memristive_model(x)
        
        # Update stats
        self.total_operations += 1
        
        # Periodically check health
        if self.total_operations % 10 == 0:
            with torch.no_grad():  # Don't affect gradients
                self.check_health()
        
        return output
    
    def visualize_system_health(self):
        """
        Visualize health of all memristive layers.
        Alias for visualize_health to maintain compatibility.
        """
        try:
            self.visualize_health()
        except Exception as e:
            print(f"Error visualizing health: {e}")
            
            # Fallback to simple visualization
            num_monitors = len(self.health_monitors)
            if num_monitors == 0:
                print("No health monitors to visualize")
                return
            
            # Just visualize overall health metrics
            health_report = self.get_health_report()
            
            plt.figure(figsize=(10, 6))
            metrics = ['overall_health', 'total_faults_detected', 'total_mitigations', 'total_operations']
            values = [health_report.get(m, 0) for m in metrics]
            
            plt.bar(metrics, values)
            plt.title('System Health Overview')
            plt.ylabel('Value')
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
    
    def check_health(self):
        """
        Check health of all memristive layers and apply mitigation if needed.
        
        Returns:
            Number of issues found
        """
        total_issues = 0
        
        # Update all health monitors
        for name, monitor in self.health_monitors.items():
            try:
                monitor.update_health_metrics()
                
                # Check for critical devices
                critical_devices = monitor.get_critical_devices()
                total_issues += len(critical_devices)
                
                if critical_devices:
                    print(f"Found {len(critical_devices)} critical devices in {name}")
                    
                    # Apply soft mitigation by regularizing conductance values
                    self._apply_soft_mitigation(name, critical_devices)
            except Exception as e:
                print(f"Error checking health for {name}: {e}")
        
        self.total_faults_detected += total_issues
        return total_issues
    
    def _apply_soft_mitigation(self, crossbar_name, critical_devices):
        """
        Apply soft mitigation to critical devices by adjusting conductance.
        
        Args:
            crossbar_name: Name of the crossbar
            critical_devices: List of (row, col) tuples for critical devices
        """
        # Get the crossbar
        name_parts = crossbar_name.split('_crossbar_')
        layer_name = name_parts[0]
        crossbar_idx = int(name_parts[1])
        
        if layer_name not in self.memristive_layers:
            return
        
        module = self.memristive_layers[layer_name]
        if not hasattr(module, 'crossbars') or crossbar_idx >= len(module.crossbars):
            return
        
        crossbar = module.crossbars[crossbar_idx]
        monitor = self.health_monitors[crossbar_name]
        
        # Get mid-range conductance
        g_mid = (monitor.g_min + monitor.g_max) / 2
        
        # Apply soft mitigation by moving conductance toward middle range
        for row, col in critical_devices:
            current_g = crossbar.conductance_matrix[row, col]
            # Move 30% of the way toward mid-range
            new_g = current_g * 0.7 + g_mid * 0.3
            crossbar.conductance_matrix[row, col] = new_g
        
        self.total_mitigations += len(critical_devices)
        print(f"Applied soft mitigation to {len(critical_devices)} devices in {crossbar_name}")
    
    def reset_critical_layer(self, layer_name):
        """
        Reset a critical layer back to its original weights if it's too degraded.
        Use as a last resort for catastrophic failures.
        
        Args:
            layer_name: Name of the layer to reset
            
        Returns:
            True if reset was successful, False otherwise
        """
        if layer_name not in self.memristive_layers or layer_name not in self.original_weights:
            return False
        
        try:
            # Get the memristive module
            module = self.memristive_layers[layer_name]
            
            # Check if module has weight parameter
            if not hasattr(module, 'weight'):
                return False
            
            # Reset weight to original value
            module.weight.data.copy_(self.original_weights[layer_name])
            
            # If module has crossbars, update them
            if hasattr(module, 'crossbars'):
                # Re-map weights to crossbars (simplistic approach)
                for crossbar in module.crossbars:
                    # This is a simplified approach - in a real implementation, 
                    # you would need to follow the same mapping routine used during initialization
                    crossbar.conductance_matrix.copy_(module.weight.data)
            
            print(f"Reset layer {layer_name} to original weights")
            return True
        
        except Exception as e:
            print(f"Error resetting layer {layer_name}: {e}")
            return False
    
    def inject_faults(self, fault_density=0.05):
        """
        Inject faults into memristive layers for testing.
        
        Args:
            fault_density: Proportion of devices to make faulty
            
        Returns:
            Number of faults injected
        """
        total_faults = 0
        
        # Inject faults into each crossbar
        for name, monitor in self.health_monitors.items():
            try:
                # Get the crossbar
                name_parts = name.split('_crossbar_')
                layer_name = name_parts[0]
                crossbar_idx = int(name_parts[1])
                
                if layer_name not in self.memristive_layers:
                    continue
                
                module = self.memristive_layers[layer_name]
                if not hasattr(module, 'crossbars') or crossbar_idx >= len(module.crossbars):
                    continue
                
                crossbar = module.crossbars[crossbar_idx]
                
                # Get total number of devices
                total_devices = crossbar.conductance_matrix.numel()
                num_faults = int(total_devices * fault_density)
                
                # Randomly select devices to fault
                indices = torch.randperm(total_devices)[:num_faults]
                rows = indices // crossbar.conductance_matrix.shape[1]
                cols = indices % crossbar.conductance_matrix.shape[1]
                
                # Inject faults - half stuck at HRS, half at LRS
                half = num_faults // 2
                
                # Stuck at HRS (low conductance)
                for i in range(half):
                    crossbar.conductance_matrix[rows[i], cols[i]] = monitor.g_min
                    # Force health score to be low
                    monitor.health_scores[rows[i], cols[i]] = 0
                
                # Stuck at LRS (high conductance)
                for i in range(half, num_faults):
                    crossbar.conductance_matrix[rows[i], cols[i]] = monitor.g_max
                    # Force health score to be low
                    monitor.health_scores[rows[i], cols[i]] = 0
                
                total_faults += num_faults
                print(f"Injected {num_faults} faults into {name}")
            
            except Exception as e:
                print(f"Error injecting faults into {name}: {e}")
        
        self.total_faults_detected += total_faults
        return total_faults
    
    def get_health_report(self, detailed=False):
        """
        Generate a comprehensive health report for the system.
        
        Args:
            detailed: Whether to include detailed information
            
        Returns:
            Dictionary with health metrics
        """
        report = {
            'total_operations': self.total_operations,
            'total_faults_detected': self.total_faults_detected,
            'total_mitigations': self.total_mitigations,
            'layer_health': {}
        }
        
        # Get health for each layer
        total_health_sum = 0
        total_devices = 0
        
        for name, monitor in self.health_monitors.items():
            try:
                health_summary = monitor.get_health_summary()
                report['layer_health'][name] = health_summary
                
                # Aggregate stats
                total_health_sum += health_summary['avg_health'] * monitor.shape[0] * monitor.shape[1]
                total_devices += monitor.shape[0] * monitor.shape[1]
            except Exception as e:
                print(f"Error getting health summary for {name}: {e}")
        
        # Calculate overall health
        if total_devices > 0:
            report['overall_health'] = total_health_sum / total_devices
        else:
            report['overall_health'] = 100.0
        
        # Add a compatible field name for avg_health since some code might expect this
        report['avg_health'] = report['overall_health']
        
        # Add placeholder for redundant_available to maintain compatibility
        report['redundant_available'] = 0
        
        return report
    
    def check_health_and_mitigate(self):
        """
        Alias for check_health to maintain compatibility with existing code.
        """
        return self.check_health()
    
    def visualize_health(self):
        """
        Visualize health of all memristive layers.
        """
        num_monitors = len(self.health_monitors)
        if num_monitors == 0:
            print("No health monitors to visualize")
            return
        
        # Determine grid size
        cols = min(3, num_monitors)
        rows = (num_monitors + cols - 1) // cols
        
        plt.figure(figsize=(5*cols, 4*rows))
        
        for i, (name, monitor) in enumerate(self.health_monitors.items()):
            plt.subplot(rows, cols, i+1)
            plt.imshow(monitor.health_scores.numpy(), cmap='RdYlGn', vmin=0, vmax=100)
            plt.colorbar(label='Health Score')
            plt.title(f'Health: {name}')
        
        plt.tight_layout()
        plt.show()

In [ ]:
# Function to run experiments with the fault-tolerant system
def run_fault_tolerance_experiment(fault_density=0.05, fault_distribution='random', 
                                  enable_mitigation=True, epochs=3, num_batches=20):
    """
    Run an experiment with the fault-tolerant neuromorphic system.
    
    Args:
        fault_density: Proportion of devices to inject faults into
        fault_distribution: How to distribute faults ('random', 'clustered', or 'gradient')
        enable_mitigation: Whether to enable fault mitigation
        epochs: Number of training epochs
        num_batches: Number of batches per epoch
        
    Returns:
        Experiment results
    """
    # Create the fault-tolerant system
    try:
        ft_system = FaultTolerantNeuromorphic(redundancy_factor=0.15)
    except Exception as e:
        print(f"Failed to create fault tolerant system: {e}")
        # Create a simplified version as fallback
        print("Creating simplified test system...")
        model = SimpleCNN()
        ft_system = FaultTolerantNeuromorphic(
            base_model=model,
            redundancy_factor=0.05  # Lower redundancy for testing
        )
    
    # Load a small subset of CIFAR-10 for testing
    transform = torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    
    batch_size = 32
    try:
        trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
        trainset = torch.utils.data.Subset(trainset, range(1000))  # Use 1000 samples
        trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)
        
        testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
        testset = torch.utils.data.Subset(testset, range(500))  # Use 500 samples
        testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False)
    except Exception as e:
        print(f"Error loading CIFAR-10 dataset: {e}")
        # Create dummy data if CIFAR-10 can't be loaded
        print("Using synthetic data instead...")
        trainset = [(torch.randn(3, 32, 32), torch.randint(0, 10, (1,)).item()) for _ in range(1000)]
        trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)
        testset = [(torch.randn(3, 32, 32), torch.randint(0, 10, (1,)).item()) for _ in range(500)]
        testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False)
    
    # Define loss and optimizer
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(ft_system.memristive_model.parameters(), lr=0.001)
    
    # Track metrics
    results = {
        'epoch': [],
        'batch': [],
        'loss': [],
        'accuracy': [],
        'health': [],
        'faults': [],
        'mitigations': []
    }
    
    # Initial system health check
    print("\nInitial health check:")
    initial_report = ft_system.get_health_report()
    print(f"Average health: {initial_report['avg_health']:.2f}%")
    
    # Inject initial faults with safe error handling
    try:
        ft_system.inject_faults(fault_density=fault_density/2, distribution=fault_distribution)
    except Exception as e:
        print(f"Warning: Failed to inject initial faults: {e}")
    
    # Training loop
    print("\nStarting training with fault injection...")
    for epoch in range(epochs):
        ft_system.memristive_model.train()
        running_loss = 0.0
        
        for batch_idx, data in enumerate(trainloader):
            if batch_idx >= num_batches:
                break
            
            try:
                # Handle both dataset types (tuple and custom)
                if isinstance(data, list) and len(data) == 2:
                    inputs, labels = data
                else:
                    inputs, labels = data
                
                # Ensure inputs have requires_grad=True for backprop
                inputs.requires_grad_(True)
                
                # Forward pass with error handling
                try:
                    outputs = ft_system.forward(inputs)
                    loss = criterion(outputs, labels)
                except Exception as e:
                    print(f"Forward pass error in batch {batch_idx}: {e}")
                    continue
                
                # Backward pass with error handling
                try:
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                except RuntimeError as e:
                    if "does not require grad" in str(e):
                        print(f"Gradient computation error in batch {batch_idx}. Skipping backpropagation.")
                        # Try to continue with the next batch
                        continue
                    else:
                        print(f"Unexpected error in backpropagation: {e}")
                        raise
                
                running_loss += loss.item()
                
                # Every few batches, inject some faults
                if batch_idx % 5 == 0 and batch_idx > 0:
                    try:
                        # Gradually increasing fault density
                        current_fault_density = fault_density * (1 + 0.1 * batch_idx / num_batches)
                        ft_system.inject_faults(fault_density=current_fault_density/10, distribution=fault_distribution)
                    except Exception as e:
                        print(f"Warning: Failed to inject faults in batch {batch_idx}: {e}")
                
                # Apply mitigation if enabled
                if enable_mitigation and batch_idx % 3 == 0:
                    try:
                        with torch.no_grad():  # Ensure mitigation doesn't affect gradients
                            ft_system.check_health_and_mitigate()
                    except Exception as e:
                        print(f"Warning: Failed to apply mitigation in batch {batch_idx}: {e}")
                
                # Evaluate current performance
                if batch_idx % 2 == 0:
                    try:
                        ft_system.memristive_model.eval()
                        correct = 0
                        total = 0
                        
                        with torch.no_grad():
                            for test_data in testloader:
                                if isinstance(test_data, list) and len(test_data) == 2:
                                    test_inputs, test_labels = test_data
                                else:
                                    test_inputs, test_labels = test_data
                                
                                test_outputs = ft_system.forward(test_inputs)
                                _, predicted = torch.max(test_outputs.data, 1)
                                total += test_labels.size(0)
                                correct += (predicted == test_labels).sum().item()
                        
                        accuracy = 100 * correct / total if total > 0 else 0
                        
                        # Get current health metrics
                        health_report = ft_system.get_health_report()
                        
                        # Record metrics
                        results['epoch'].append(epoch)
                        results['batch'].append(batch_idx)
                        results['loss'].append(running_loss / (batch_idx + 1))
                        results['accuracy'].append(accuracy)
                        results['health'].append(health_report['avg_health'])
                        results['faults'].append(health_report['total_faults_detected'])
                        results['mitigations'].append(health_report['total_mitigations'])
                        
                        print(f"Epoch {epoch+1}, Batch {batch_idx+1}, Loss: {running_loss/(batch_idx+1):.4f}, Accuracy: {accuracy:.2f}%")
                        print(f"  Health: {health_report['avg_health']:.2f}%, Faults: {health_report['total_faults_detected']}, Mitigations: {health_report['total_mitigations']}")
                        
                        ft_system.memristive_model.train()
                    except Exception as e:
                        print(f"Error during evaluation in batch {batch_idx}: {e}")
            
            except Exception as e:
                print(f"Unexpected error in batch {batch_idx}: {e}")
                continue
    
    # Final health check
    print("\nFinal health check:")
    final_report = ft_system.get_health_report()
    print(f"Average health: {final_report['avg_health']:.2f}%")
    print(f"Total faults: {final_report['total_faults_detected']}")
    print(f"Total mitigations: {final_report['total_mitigations']}")
    print(f"Redundant devices available: {final_report['redundant_available']}")
    
    # Visualize system health (optional)
    try:
        ft_system.visualize_system_health()
    except Exception as e:
        print(f"Failed to visualize system health: {e}")
    
    # Plot results (optional)
    try:
        plt.figure(figsize=(15, 10))
        
        # Plot accuracy vs health
        plt.subplot(2, 2, 1)
        plt.plot(results['batch'], results['accuracy'], 'b-', label='Accuracy')
        plt.ylabel('Accuracy (%)', color='b')
        plt.title('Accuracy vs Health')
        
        ax2 = plt.twinx()
        ax2.plot(results['batch'], results['health'], 'r-', label='Health')
        ax2.set_ylabel('Health (%)', color='r')
        
        # Plot faults and mitigations
        plt.subplot(2, 2, 2)
        plt.plot(results['batch'], results['faults'], 'r-', label='Faults')
        plt.plot(results['batch'], results['mitigations'], 'g-', label='Mitigations')
        plt.title('Faults vs Mitigations')
        plt.xlabel('Batch')
        plt.ylabel('Count')
        plt.legend()
        
        # Plot loss
        plt.subplot(2, 2, 3)
        plt.plot(results['batch'], results['loss'], 'b-')
        plt.title('Training Loss')
        plt.xlabel('Batch')
        plt.ylabel('Loss')
        
        # Plot accuracy over time
        plt.subplot(2, 2, 4)
        plt.plot(results['batch'], results['accuracy'], 'g-')
        plt.title('Test Accuracy')
        plt.xlabel('Batch')
        plt.ylabel('Accuracy (%)')
        
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Failed to plot results: {e}")
    
    return {
        'system': ft_system,
        'metrics': results,
        'final_report': final_report
    }

In [ ]:
def main():
    """
    Main function to demonstrate the fault-tolerant neuromorphic computing system
    with proper error handling
    """
    try:
        print("Running fault tolerance experiment with mitigation enabled...")
        results_with_mitigation = run_fault_tolerance_experiment(
            fault_density=0.1,
            fault_distribution='random',
            enable_mitigation=True,
            epochs=2,
            num_batches=10
        )
        
        # Check if results were successfully generated
        mitigation_success = (
            isinstance(results_with_mitigation, dict) and
            'metrics' in results_with_mitigation and
            'accuracy' in results_with_mitigation['metrics'] and
            len(results_with_mitigation['metrics']['accuracy']) > 0
        )
        
        print("\nRunning fault tolerance experiment with mitigation disabled...")
        results_without_mitigation = run_fault_tolerance_experiment(
            fault_density=0.1,
            fault_distribution='random',
            enable_mitigation=False,
            epochs=2,
            num_batches=10
        )
        
        # Check if results were successfully generated
        no_mitigation_success = (
            isinstance(results_without_mitigation, dict) and
            'metrics' in results_without_mitigation and
            'accuracy' in results_without_mitigation['metrics'] and
            len(results_without_mitigation['metrics']['accuracy']) > 0
        )
        
        # Compare results with safe access
        print("\nComparison of results:")
        
        print("With mitigation:")
        if mitigation_success:
            print(f"  Final accuracy: {results_with_mitigation['metrics']['accuracy'][-1]:.2f}%")
            print(f"  Final health: {results_with_mitigation['metrics']['health'][-1]:.2f}%")
            print(f"  Total faults: {results_with_mitigation['final_report']['total_faults_detected']}")
            print(f"  Total mitigations: {results_with_mitigation['final_report']['total_mitigations']}")
        else:
            print("  No valid results were generated with mitigation enabled.")
        
        print("\nWithout mitigation:")
        if no_mitigation_success:
            print(f"  Final accuracy: {results_without_mitigation['metrics']['accuracy'][-1]:.2f}%")
            print(f"  Final health: {results_without_mitigation['metrics']['health'][-1]:.2f}%")
            print(f"  Total faults: {results_without_mitigation['final_report']['total_faults_detected']}")
            print(f"  Total mitigations: {results_without_mitigation['final_report']['total_mitigations']}")
        else:
            print("  No valid results were generated with mitigation disabled.")
        
        # If neither experiment worked, print a helpful message
        if not (mitigation_success or no_mitigation_success):
            print("\nBoth experiments failed to generate valid results. This likely indicates:")
            print("1. There may be compatibility issues with the MemTorch library")
            print("2. The shape mismatch in the crossbar matrices is causing persistent errors")
            print("3. The gradient computation issues weren't fully resolved")
            print("\nConsider running a smaller test case to debug further.")
    
    except Exception as e:
        print(f"\nAn error occurred in the main function: {str(e)}")
        import traceback
        traceback.print_exc()
        print("\nThe experiment failed to complete. This might be due to:")
        print("1. Issues with the PyTorch or MemTorch library compatibility")
        print("2. Memory constraints when creating large crossbar arrays")
        print("3. Unresolved index bounds issues in the fault injection or self-healing code")
        print("\nYou might want to try with a smaller network or reduced redundancy factor.")


if __name__ == "__main__":
    main()